# 02 - Prepare Condition Data

Build compact SFT records for `product + reactants + reaction_class -> conditions` from ORD. This notebook filters out records without useful condition labels.

In [ ]:
!pip install -q -U huggingface_hub ord-schema protobuf rdkit tqdm

import gzip
import json
import random
from pathlib import Path
from typing import Any, Iterable, Optional

from google.protobuf.json_format import MessageToDict
from huggingface_hub import HfApi, hf_hub_download
from ord_schema.proto import dataset_pb2
from rdkit import Chem, RDLogger
from tqdm.auto import tqdm

RDLogger.DisableLog('rdApp.warning')
RDLogger.DisableLog('rdApp.error')

SEED = 3407
HF_DATASET_ID = 'open-reaction-database/ord-data'
MAX_PB_FILES = 120
MAX_ROWS = 40000
EVAL_FRACTION = 0.05
TEST_FRACTION = 0.05

WORK_DIR = Path('/content/retro_condition_agent')
ORD_DIR = WORK_DIR / 'ord_data'
DATA_DIR = WORK_DIR / 'data'
ORD_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'conditions_train.jsonl'
EVAL_PATH = DATA_DIR / 'conditions_eval.jsonl'
TEST_PATH = DATA_DIR / 'conditions_test.jsonl'
ROWS_PATH = DATA_DIR / 'conditions_rows.jsonl'
random.seed(SEED)


In [ ]:
SYSTEM_PROMPT = '''You are a chemistry condition recommendation model.
Return valid compact JSON only.
Given product SMILES, reactants, and reaction class, predict practical reaction conditions.
Do not invent evidence IDs and do not include explanations.'''

def canonical_smiles(smiles: Optional[str]) -> Optional[str]:
    if not smiles:
        return None
    try:
        mol = Chem.MolFromSmiles(str(smiles).strip())
        if mol is None:
            return None
        for atom in mol.GetAtoms():
            atom.SetAtomMapNum(0)
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None

def has_wildcard_smiles(smiles: str) -> bool:
    return '*' in str(smiles)

def useful_value(value: str) -> bool:
    return value not in (None, '', 'not specified')

def unique_keep_order(values: Iterable[str]) -> list[str]:
    out, seen = [], set()
    for value in values:
        value = str(value).strip() if value is not None else ''
        if not value or value.lower() in {'none', 'null', 'nan', 'not specified'}:
            continue
        if value not in seen:
            seen.add(value)
            out.append(value)
    return out

def first_identifier(obj: dict, identifier_type: str) -> Optional[str]:
    wanted = identifier_type.upper()
    for ident in obj.get('identifiers', []) or []:
        if str(ident.get('type', '')).upper() == wanted and ident.get('value'):
            return str(ident['value']).strip()
    return None

def first_smiles(obj: dict) -> Optional[str]:
    return canonical_smiles(first_identifier(obj, 'SMILES'))

def first_name(obj: dict) -> Optional[str]:
    return first_identifier(obj, 'NAME')

def format_quantity(q: Any) -> str:
    if not isinstance(q, dict):
        return 'not specified'
    value = q.get('value')
    units = q.get('units')
    if value is None:
        return 'not specified'
    return f'{value} {str(units).lower()}' if units else str(value)

def extract_inputs(reaction: dict) -> dict:
    reactants, solvents, reagents = [], [], []
    for reaction_input in (reaction.get('inputs') or {}).values():
        for component in reaction_input.get('components', []) or []:
            role = str(component.get('reaction_role', '')).upper().strip()
            smiles = first_smiles(component)
            label = first_name(component) or smiles
            if role in {'REACTANT', 'STARTING_MATERIAL', 'SUBSTRATE'} and smiles:
                reactants.append(smiles)
            elif role == 'SOLVENT' and label:
                solvents.append(label)
            elif role not in {'PRODUCT', 'INTERNAL_STANDARD'} and label:
                reagents.append(label)
    return {'reactants': unique_keep_order(reactants), 'solvents': unique_keep_order(solvents), 'reagents': unique_keep_order(reagents)}

def extract_yield_percent(product: dict) -> Optional[float]:
    values = []
    for measurement in product.get('measurements', []) or []:
        if str(measurement.get('type', '')).upper() != 'YIELD':
            continue
        value = (measurement.get('percentage') or {}).get('value')
        try:
            if value is not None:
                values.append(float(value))
        except Exception:
            pass
    return max(values) if values else None

def extract_best_product(reaction: dict) -> Optional[dict]:
    candidates = []
    for outcome in reaction.get('outcomes', []) or []:
        reaction_time = format_quantity(outcome.get('reaction_time'))
        for product in outcome.get('products', []) or []:
            role = str(product.get('reaction_role', '')).upper().strip()
            if role and role != 'PRODUCT':
                continue
            smiles = first_smiles(product)
            if not smiles:
                continue
            y = extract_yield_percent(product)
            score = (10000 if product.get('is_desired_product') else 0) + (1000 + y if y is not None else 0)
            candidates.append({'product_smiles': smiles, 'reaction_time': reaction_time, 'yield_percent': y, 'score': score})
    return sorted(candidates, key=lambda x: x['score'], reverse=True)[0] if candidates else None

def extract_workup(reaction: dict) -> str:
    parts = []
    for workup in reaction.get('workups', []) or []:
        wtype = workup.get('type')
        details = workup.get('details')
        if wtype and details:
            parts.append(f'{wtype}: {details}')
        elif details or wtype:
            parts.append(str(details or wtype))
    return '; '.join(parts) if parts else 'not specified'

def extract_row(reaction: dict) -> Optional[dict]:
    inputs = extract_inputs(reaction)
    product = extract_best_product(reaction)
    if not product or not inputs['reactants']:
        return None
    if has_wildcard_smiles(product['product_smiles']) or any(has_wildcard_smiles(r) for r in inputs['reactants']):
        return None
    conditions = reaction.get('conditions') or {}
    temperature = format_quantity((conditions.get('temperature') or {}).get('setpoint'))
    atmosphere = str(((reaction.get('setup') or {}).get('environment') or {}).get('details') or 'not specified')
    y = product['yield_percent']
    row = {
        'reaction_id': str(reaction.get('reaction_id') or reaction.get('id') or ''),
        'product_smiles': product['product_smiles'],
        'reactants': inputs['reactants'],
        'reaction_class': 'unknown',
        'reagents': '; '.join(inputs['reagents']) if inputs['reagents'] else 'not specified',
        'solvent': '; '.join(inputs['solvents']) if inputs['solvents'] else 'not specified',
        'temperature_celsius': temperature,
        'reaction_time': product['reaction_time'],
        'atmosphere': atmosphere,
        'workup_purification': extract_workup(reaction),
        'expected_yield_percent': 'not specified' if y is None else f'{round(float(y), 1)}%',
    }
    output_condition_fields = ['reagents', 'solvent', 'temperature_celsius', 'reaction_time', 'atmosphere', 'workup_purification', 'expected_yield_percent']
    useful = any(useful_value(row[k]) for k in output_condition_fields)
    return row if useful else None


In [ ]:
def iter_ord_reactions(path: Path):
    dataset = dataset_pb2.Dataset()
    with gzip.open(path, 'rb') as f:
        dataset.ParseFromString(f.read())
    for reaction_proto in dataset.reactions:
        yield MessageToDict(reaction_proto, preserving_proto_field_name=True)

def make_record(row: dict) -> dict:
    user = json.dumps({
        'task': 'predict_conditions',
        'product_smiles': row['product_smiles'],
        'reactants': row['reactants'],
        'reaction_class': row['reaction_class'],
    }, separators=(',', ':'))
    assistant = json.dumps({
        'reagents': row['reagents'],
        'solvent': row['solvent'],
        'temperature_celsius': row['temperature_celsius'],
        'reaction_time': row['reaction_time'],
        'atmosphere': row['atmosphere'],
        'workup_purification': row['workup_purification'],
        'expected_yield_percent': row['expected_yield_percent'],
    }, separators=(',', ':'))
    return {'messages': [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user}, {'role': 'assistant', 'content': assistant}], 'metadata': {'reaction_id': row['reaction_id'], 'source': 'ORD'}}

def write_jsonl(path: Path, records: list[dict]) -> None:
    with path.open('w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

api = HfApi()
all_files = sorted([f for f in api.list_repo_files(HF_DATASET_ID, repo_type='dataset') if f.startswith('data/') and f.endswith('.pb.gz')])
random.shuffle(all_files)
selected = all_files[:MAX_PB_FILES]
local_files = [Path(hf_hub_download(HF_DATASET_ID, filename=f, repo_type='dataset', local_dir=str(ORD_DIR))) for f in tqdm(selected, desc='download ORD')]

rows = []
for path in tqdm(local_files, desc='parse ORD'):
    for reaction in iter_ord_reactions(path):
        row = extract_row(reaction)
        if row:
            rows.append(row)

random.shuffle(rows)
if MAX_ROWS is not None:
    rows = rows[:MAX_ROWS]

n_total = len(rows)
n_test = max(1, int(n_total * TEST_FRACTION))
n_eval = max(1, int(n_total * EVAL_FRACTION))
test_rows = rows[:n_test]
eval_rows = rows[n_test:n_test + n_eval]
train_rows = rows[n_test + n_eval:]

write_jsonl(TRAIN_PATH, [make_record(r) for r in train_rows])
write_jsonl(EVAL_PATH, [make_record(r) for r in eval_rows])
write_jsonl(TEST_PATH, [make_record(r) for r in test_rows])
write_jsonl(ROWS_PATH, rows)
print('rows:', n_total)
print('train/eval/test:', len(train_rows), len(eval_rows), len(test_rows))
print('saved:', TRAIN_PATH, EVAL_PATH, TEST_PATH)
